# Instrucciones

**Carga del conjunto de datos.** Usaremos el dataset **Adult Income Dataset**, también conocido como **"Census Income"** este información fue recolectada por la Oficina del Censo de EE.UU. y descargada por la academia para guardarla en esta carpeta de proyecto bajo el nombre `adult-census-income.csv` o puedes cargarlo en el código directamente desde el siguente enlace:
https://breathecode.herokuapp.com/asset/internal-link?id=2326&path=adult-census-income.csv

Este dataset incluye variables como:
* Edad
* Nivel educativo
* Estado civil
* Ocupación
* Horas trabajadas por semana
* Sexo
* País de origen
* Ingreso anual (>50K o <=50K)

**Preprocesamiento de datos.** Haz la limpieza de datos nulos o mal codificados, la transformación de variables categóricas, normaliza las variables numéricas.

**Define el problema de recomendación.** Plantea cómo vas a estructurar tu sistema de recomendación:
* ¿Qué se quiere recomendar?
* ¿Cuál será el "usuario" en este caso?
* ¿Qué variables definen el perfil de un usuario?

**Construye el sistema de recomendación.** Usa uno de los siguientes enfoques:
1. **Filtrado basado en contenido.** Representa a cada usuario como un vector y calcula similitudes entre usuarios y recomendaciones.
2. **Filtrado colaborativo.** Simula una matriz de usuarios vs. trayectorias. Aplica k-NN, correlación de Pearson o matrix factorization.
3. **Sistema híbrido.** Combina ambos enfoques.

**Pruebas con casos simulados.** Construye perfiles simulados de usuarios hipotéticos y fijate qué trayectorias (educación, ocupación, etc.) les recomendaría el sistema para mejorar su ingreso estimado.

**# Ejemplo: Usuario de 25 años, secundario completo, trabaja medio tiempo**
`perfil_usuario = {...}`

In [63]:
import os
import joblib
import regex as re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold, GridSearchCV
from sklearn.svm import SVC
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
from collections import Counter
from imblearn.over_sampling import SMOTE
from scipy.stats import loguniform
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neighbors import NearestNeighbors


In [64]:
# Cargamos es dataset
metadata = pd.read_csv('../data/raw/adult-census-income.csv', low_memory = False)
metadata.head(3)

,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
0,90,?,77053,HS-grad,9,Widowed,?,Not-in-family,White,Female,0,4356,40,United-States,<=50K
1,82,Private,132870,HS-grad,9,Widowed,Exec-managerial,Not-in-family,White,Female,0,4356,18,United-States,<=50K
2,66,?,186061,Some-college,10,Widowed,?,Unmarried,Black,Female,0,4356,40,United-States,<=50K


In [65]:
# Reemplazamos '?' por NaN para que pandas los reconozca
metadata.replace('?', np.nan, inplace=True)

,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
0,90,NaN,77053,HS-grad,9,Widowed,NaN,Not-in-family,White,Female,0,4356,40,United-States,<=50K
1,82,Private,132870,HS-grad,9,Widowed,Exec-managerial,Not-in-family,White,Female,0,4356,18,United-States,<=50K
2,66,NaN,186061,Some-college,10,Widowed,NaN,Unmarried,Black,Female,0,4356,40,United-States,<=50K
3,54,Private,140359,7th-8th,4,Divorced,Machine-op-inspct,Unmarried,White,Female,0,3900,40,United-States,<=50K
4,41,Private,264663,Some-college,10,Separated,Prof-specialty,Own-child,White,Female,0,3900,40,United-States,<=50K
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32556,22,Private,310152,Some-college,10,Never-married,Protective-serv,Not-in-family,White,Male,0,0,40,United-States,<=50K
32557,27,Private,257302,Assoc-acdm,12,Married-civ-spouse,Tech-support,Wife,White,Female,0,0,38,United-States,<=50K
32558,40,Private,154374,HS-grad,9,Married-civ-spouse,Machine-op-inspct,Husband,White,Male,0,0,40,United-States,>50K
32559,58,Private,151910,HS-grad,9,Widowed,Adm-clerical,Unmarried,White,Female,0,0,40,United-States,<=50K


In [66]:
# Verificamos las columnas
metadata.info()

<class 'pandas.DataFrame'>
RangeIndex: 32561 entries, 0 to 32560
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   age             32561 non-null  int64
 1   workclass       30725 non-null  str  
 2   fnlwgt          32561 non-null  int64
 3   education       32561 non-null  str  
 4   education.num   32561 non-null  int64
 5   marital.status  32561 non-null  str  
 6   occupation      30718 non-null  str  
 7   relationship    32561 non-null  str  
 8   race            32561 non-null  str  
 9   sex             32561 non-null  str  
 10  capital.gain    32561 non-null  int64
 11  capital.loss    32561 non-null  int64
 12  hours.per.week  32561 non-null  int64
 13  native.country  31978 non-null  str  
 14  income          32561 non-null  str  
dtypes: int64(6), str(9)
memory usage: 3.7 MB


In [67]:
# Estadisticas descriptivas
metadata.describe()

,age,fnlwgt,education.num,capital.gain,capital.loss,hours.per.week
count,32561.000000,3.256100e+04,32561.000000,32561.000000,32561.000000,32561.000000
mean,38.581647,1.897784e+05,10.080679,1077.648844,87.303830,40.437456
std,13.640433,1.055500e+05,2.572720,7385.292085,402.960219,12.347429
min,17.000000,1.228500e+04,1.000000,0.000000,0.000000,1.000000
25%,28.000000,1.178270e+05,9.000000,0.000000,0.000000,40.000000
50%,37.000000,1.783560e+05,10.000000,0.000000,0.000000,40.000000
75%,48.000000,2.370510e+05,12.000000,0.000000,0.000000,45.000000
max,90.000000,1.484705e+06,16.000000,99999.000000,4356.000000,99.000000


In [68]:
# Verificamos cuantos nulos hay por columna
print(metadata.isnull().sum())

age                  0
workclass         1836
fnlwgt               0
education            0
education.num        0
marital.status       0
occupation        1843
relationship         0
race                 0
sex                  0
capital.gain         0
capital.loss         0
hours.per.week       0
native.country     583
income               0
dtype: int64


## Analisis hasta ahora
* Tenemos un dataframe de 32561 filas y 15 columnas
* No tiene identificador unico por lo que podemos asumir que al menos por esta parte no hay duplicados.
* Las columnas son 9 str y 6 int64.
* Habian valores con "?" las reemplazamos con Nulos PAra que pandas los pueda interpretar.
* `capital.gain` y `capital.loss` tiene outliers muy agresivos.

### Ahora
Tenemos que plantearnos 3 cosas segun el enunciado:
**¿Que se quiere recomendar?**: En este contexto, lo mas logico seria recomendar "Trayectorias educativas o profecionales para mejorar". Es decir, cambios en la Ocupación o el Nivel Educativo que historicamente han llevado a las personas a ganar mas de 50K.

**¿Cuál sera el "usuario"?**: Cualquier persona del dataset que actualmente gane <=50K o incluso fuera del mismo si el despliegue del modelo lo permite.

**¿Qué variables definen el perfil?**: En primera instancia entre las que ya tenemos: edad, sexo, pais, horas trabajadas, etc, Pero aun falta procesamiento factorizacion,dummies,etc.

In [69]:
# Vamos a imputar por el valor mas frecuente, la moda
for col in metadata.columns:
    if metadata[col].isnull().any():
        # Obtenemos la moda de la columna y tomamos el primer valor
        moda = metadata[col].mode()[0]
        metadata[col] = metadata[col].fillna(moda)
metadata

,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
0,90,Private,77053,HS-grad,9,Widowed,Prof-specialty,Not-in-family,White,Female,0,4356,40,United-States,<=50K
1,82,Private,132870,HS-grad,9,Widowed,Exec-managerial,Not-in-family,White,Female,0,4356,18,United-States,<=50K
2,66,Private,186061,Some-college,10,Widowed,Prof-specialty,Unmarried,Black,Female,0,4356,40,United-States,<=50K
3,54,Private,140359,7th-8th,4,Divorced,Machine-op-inspct,Unmarried,White,Female,0,3900,40,United-States,<=50K
4,41,Private,264663,Some-college,10,Separated,Prof-specialty,Own-child,White,Female,0,3900,40,United-States,<=50K
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32556,22,Private,310152,Some-college,10,Never-married,Protective-serv,Not-in-family,White,Male,0,0,40,United-States,<=50K
32557,27,Private,257302,Assoc-acdm,12,Married-civ-spouse,Tech-support,Wife,White,Female,0,0,38,United-States,<=50K
32558,40,Private,154374,HS-grad,9,Married-civ-spouse,Machine-op-inspct,Husband,White,Male,0,0,40,United-States,>50K
32559,58,Private,151910,HS-grad,9,Widowed,Adm-clerical,Unmarried,White,Female,0,0,40,United-States,<=50K


In [70]:
#Vamos a guardas listas de columnas por si nos hacen falta mas adelante
model_columns = metadata.drop(columns = ['income']).columns
numeric_columns = metadata[model_columns].select_dtypes(include = ['int', 'float']).columns
categorical_columns = metadata[model_columns].select_dtypes(include = ['object', 'string']).columns
columns = {
    'model':model_columns,
    'numeric':numeric_columns,
    'categorical':categorical_columns,
    }

In [71]:
# Definimos una funcion para preparar el dataset y datos para probar el modelo
def prepare_user_data(user_data, columns, fitted_scaler = False):
    # Primero que nada copiamos el dataset y controlamos los outliers de "capital" con log1p
    user_df = pd.DataFrame([user_data]).copy() if not isinstance(user_data, pd.DataFrame) else user_data.copy()
    user_df["capital.gain"] = np.log1p(user_df["capital.gain"])
    user_df["capital.loss"] = np.log1p(user_df["capital.loss"])
    #fitted_scaler es simplemente usamos un escalador nuevo porque estamos entrenando o por el contrario usamos el que ya esta porque solo vamos a probar el modelo.
    if fitted_scaler == False:
        if 'income' in user_df.columns:
            #Hacemos un  labelencoder en income pero sin la libreria ya que para este caso solo hace falta hacer un map.
            user_df['income'] = user_df['income'].map({'<=50K': 0, '>50K': 1})
        #Escalamos
        scaler = StandardScaler()
        user_df[columns["numeric"]] = scaler.fit_transform(user_df[columns["numeric"]])
        #Y generamos los dummies o onehot encoding para las variables categoricas
        user_df = pd.get_dummies(user_df, columns= columns["categorical"])
        return user_df, scaler
    else:
        # Si ya tenemos un scaler y solo queremos procesar datos para probar el modelo solo generamos los dummies y reindexamos
        user_df = pd.get_dummies(user_df)
        user_df = user_df.reindex(columns=columns["processed"], fill_value=0)
        user_df[columns["numeric"]] = fitted_scaler.transform(user_df[columns["numeric"]])
        return user_df


In [ ]:
# Aplicamos la funcion que hicimos
metadata_processed, scaler = prepare_user_data(metadata, columns = columns)
columns["processed"] = metadata_processed.columns
metadata_processed.head()

,age,fnlwgt,education.num,capital.gain,capital.loss,hours.per.week,income,workclass_Federal-gov,workclass_Local-gov,workclass_Never-worked,...,native.country_Portugal,native.country_Puerto-Rico,native.country_Scotland,native.country_South,native.country_Taiwan,native.country_Thailand,native.country_Trinadad&Tobago,native.country_United-States,native.country_Vietnam,native.country_Yugoslavia
0,3.769612,-1.067997,-0.420060,-0.299271,5.067180,-0.035429,0,False,False,False,...,False,False,False,False,False,False,False,True,False,False
1,3.183112,-0.539169,-0.420060,-0.299271,5.067180,-1.817204,0,False,False,False,...,False,False,False,False,False,False,False,True,False,False
2,2.010110,-0.035220,-0.031360,-0.299271,5.067180,-0.035429,0,False,False,False,...,False,False,False,False,False,False,False,True,False,False
3,1.130359,-0.468215,-2.363558,-0.299271,4.997412,-0.035429,0,False,False,False,...,False,False,False,False,False,False,False,True,False,False
4,0.177296,0.709482,-0.031360,-0.299271,4.997412,-0.035429,0,False,False,False,...,False,False,False,False,False,False,False,True,False,False


In [ ]:
def recommend_content(user_profile_df, df_processed, df_original, n_recommendations=5):
    # primero filtramos los perfiles con income >50K en el dataset procesado
    profiles = df_processed[df_processed['income'] == 1].copy()
    # Extraemos solo las columnas de características (sin income)
    features = profiles.drop(columns=['income'])
    # Nos aseguramos que el perfil tenga las mismas columnas y el mismo orden
    user_features = user_profile_df[features.columns]
    # Calcular la similitud usando "cosine_similarity"
    similarities = cosine_similarity(user_features, features)
    # Obtenemos los indices de los perfiles más parecidos
    top = similarities[0].argsort()[-n_recommendations:][::-1]
    # Obtener los indices reales del DataFrame original (el que no está reescalado)
    real_indices = profiles.iloc[top].index
    # Devolver las recomendaciones interpretables
    return df_original.loc[real_indices]


In [74]:
# Vamos a declarar algunos usuario des prueba
test_user_list = [{
    'age': 25,
    'workclass': 'Local-gov',
    'fnlwgt': 226802,
    'education': 'HS-grad',
    'education.num': 9,
    'marital.status': 'Never-married',
    'occupation': 'Machine-op-inspct',
    'relationship': 'Own-child',
    'race': 'White',
    'sex': 'Male',
    'capital.gain': 0,
    'capital.loss': 0,
    'hours.per.week': 20, 
    'native.country': 'United-States'
},
{
    'age': 42,
    'workclass': 'Private',
    'fnlwgt': 120000,
    'education': 'Bachelors',
    'education.num': 13,
    'marital.status': 'Married-civ-spouse',
    'occupation': 'Exec-managerial',
    'relationship': 'Husband',
    'race': 'White',
    'sex': 'Male',
    'capital.gain': 15000,
    'capital.loss': 0,
    'hours.per.week': 50,
    'native.country': 'United-States'
},
{
    'age': 32,
    'workclass': 'Private',
    'fnlwgt': 150000,
    'education': 'Some-college',
    'education.num': 9,
    'marital.status': 'Never-married',
    'occupation': 'Tech-support',
    'relationship': 'Not-in-family',
    'race': 'Black',
    'sex': 'Female',
    'capital.gain': 1000,
    'capital.loss': 200,
    'hours.per.week': 30,
    'native.country': 'United-States'
}
]

In [ ]:
# hacemos un bucle por cada usuario de prueba que tenemos y le aplicamos el prepare_user_data
test_user_list_prepared=[]
for user in test_user_list:
    test_user_list_prepared.append(prepare_user_data(user, columns = columns, fitted_scaler = scaler))
  
test_user_list_prepared = pd.concat(test_user_list_prepared, ignore_index = True)
test_user_list_prepared


,age,fnlwgt,education.num,capital.gain,capital.loss,hours.per.week,income,workclass_Federal-gov,workclass_Local-gov,workclass_Never-worked,...,native.country_Portugal,native.country_Puerto-Rico,native.country_Scotland,native.country_South,native.country_Taiwan,native.country_Thailand,native.country_Trinadad&Tobago,native.country_United-States,native.country_Vietnam,native.country_Yugoslavia
0,-0.995706,0.350774,-0.420060,-0.299271,-0.221075,-1.655225,0,0,1,0,...,0,0,0,0,0,0,0,True,0,0
1,0.250608,-0.661103,1.134739,3.618060,-0.221075,0.774468,0,0,0,0,...,0,0,0,0,0,0,0,True,0,0
2,-0.482518,-0.376873,-0.420060,2.515229,3.125795,-0.845327,0,0,0,0,...,0,0,0,0,0,0,0,True,0,0


In [ ]:
# Ahora volvemos a hacer un bucle pero para imprimir las recomendaciones
print("--- TRAYECTORIAS RECOMENDADAS PARA EL PERFIL(POR CONTENIDO) ---")
for i in range(len(test_user_list_prepared)):
    aux = test_user_list_prepared[i : i + 1].copy()
    recommended_profiles = recommend_content(aux, metadata_processed, metadata)
    print(f"Recomendaciones Usuario N° {i + 1}")
    display(recommended_profiles)


--- TRAYECTORIAS RECOMENDADAS PARA EL PERFIL(POR CONTENIDO) ---
Recomendaciones Usuario N° 1


,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
8706,25,Private,159662,HS-grad,9,Married-civ-spouse,Sales,Own-child,White,Male,0,0,26,United-States,>50K
24199,23,Local-gov,197918,Some-college,10,Never-married,Protective-serv,Own-child,White,Male,0,0,40,United-States,>50K
25206,39,Local-gov,236391,HS-grad,9,Married-civ-spouse,Machine-op-inspct,Husband,White,Male,0,0,38,United-States,>50K
20651,32,Private,209103,HS-grad,9,Married-civ-spouse,Transport-moving,Husband,White,Male,0,0,20,United-States,>50K
24514,22,Private,127768,Some-college,10,Never-married,Other-service,Own-child,White,Male,0,0,32,United-States,>50K


Recomendaciones Usuario N° 2


,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
1976,43,Private,130126,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,15024,0,50,United-States,>50K
1928,40,Private,88909,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,15024,0,50,United-States,>50K
2425,41,Private,151504,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,7688,0,50,United-States,>50K
2747,44,Private,145441,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,7298,0,48,United-States,>50K
2883,46,Private,98637,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,7298,0,50,United-States,>50K


Recomendaciones Usuario N° 3


,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
53,42,Private,107276,Some-college,10,Never-married,Exec-managerial,Not-in-family,White,Female,0,2444,40,United-States,>50K
18,22,Private,119592,Assoc-acdm,12,Never-married,Handlers-cleaners,Not-in-family,Black,Male,0,2824,40,United-States,>50K
1339,45,Private,196584,Assoc-voc,11,Never-married,Prof-specialty,Not-in-family,White,Female,0,1564,40,United-States,>50K
801,27,Private,169117,HS-grad,9,Married-civ-spouse,Adm-clerical,Wife,Black,Female,0,1887,40,United-States,>50K
198,27,Local-gov,92431,Some-college,10,Never-married,Protective-serv,Not-in-family,White,Male,0,2231,40,United-States,>50K


## Vale que tenemos?
`Ya tenmos los qu nos arroja el recomendador por ahora solo explicaremos al usuario 1 ya que igualmente esta incompleto el recomendador`

**Podemos saca estas sugerencias utiles para el usuario 1:**

* Seguir estudiando un poco mas por el "Some-college" de los usuarios similares.
* Considerar cambiar de operador de maquina a Ventas o Transporte": Para alguien con secundaria completa es totalmente viable.
* Aumentar las horas o buscar eficiencia: Nuestro usuario tiene 20, los usuarios similares tienen > 20 es indicador clave diria yo.


In [ ]:
# Ahora vamos a definir el modelo de recomendacion knn colaborativo
def recommend_knn(user_profile_df, df_processed, df_original, k = 5):
    
    # entrenar KNN con todos los usuarios
    knn_model = NearestNeighbors(metric = 'cosine', algorithm='brute')
    knn_model.fit(df_processed)

    # buscamos vecinos
    _ , indices = knn_model.kneighbors(user_profile_df, n_neighbors = k)

    # obtenemos indices reales
    neighbor_indices = indices.flatten()

    recommendations = df_original.iloc[neighbor_indices]

    if len(recommendations[recommendations['income'] == '>50K']) > 0:
        return recommendations[recommendations['income'] == '>50K']
    else:
        return recommendations

In [ ]:
# Volvemos a hacer el bloque pero con knn
print("--- TRAYECTORIAS RECOMENDADAS PARA EL PERFIL(KNN COLABORATIVO) ---")
for i in range(len(test_user_list_prepared)):
    aux = test_user_list_prepared[i : i + 1].copy()
    recommended_profiles = recommend_knn( aux, metadata_processed, metadata)
    display(recommended_profiles[['age', 'education', 'occupation', 'hours.per.week', 'capital.gain','capital.loss', 'income']])


--- TRAYECTORIAS RECOMENDADAS PARA EL PERFIL(KNN COLABORATIVO) ---


,age,education,occupation,hours.per.week,capital.gain,capital.loss,income
24111,21,HS-grad,Machine-op-inspct,20,0,0,<=50K
26833,22,HS-grad,Farming-fishing,20,0,0,<=50K
8856,19,HS-grad,Machine-op-inspct,24,0,0,<=50K
13322,22,HS-grad,Machine-op-inspct,24,0,0,<=50K
13116,21,HS-grad,Machine-op-inspct,30,0,0,<=50K


,age,education,occupation,hours.per.week,capital.gain,capital.loss,income
1976,43,Bachelors,Exec-managerial,50,15024,0,>50K
1928,40,Bachelors,Exec-managerial,50,15024,0,>50K
2020,42,Bachelors,Exec-managerial,45,15024,0,>50K


,age,education,occupation,hours.per.week,capital.gain,capital.loss,income
1103,30,Some-college,Prof-specialty,25,0,1719,<=50K
1085,20,HS-grad,Adm-clerical,30,0,1721,<=50K
1248,18,Some-college,Handlers-cleaners,35,0,1602,<=50K
1097,23,Some-college,Adm-clerical,40,0,1721,<=50K
307,25,Assoc-acdm,Tech-support,30,0,2001,<=50K


## Un poco diferente no?
El knn colaborativo nos arroja resultados de income <=50 para el usuario 1 y 3, pero esto puede tener una explicacion, mientras que nuestro recomendador anterior buscaba similitudes en palabras de una forma mas basica, el knn es mas estricto o realistas("busca vecinos mas cercanos pues") por lo que solo el usuario 2 que tenia un capital.gain que no es de 0 si les arrojo recomendaciones realistas.

In [ ]:
def hybrid_recommendation(user_profile_df, processed_metadata_df, original_metadata_df, n_content=5, k_knn=5, weight_content=0.5, weight_knn=0.5):
    # Obtenemos las recomendaciones por contenido
    content_recommendations = recommend_content(user_profile_df, processed_metadata_df, original_metadata_df, n_recommendations=n_content)
    content_scores = {index: n_content - rank for rank, index in enumerate(content_recommendations.index)}

    # Obtenemos las eecomendaciones por colaborativo
    collaborative_recommendations = recommend_knn(user_profile_df, processed_metadata_df, original_metadata_df, k=k_knn)
    collaborative_scores = {index: k_knn - rank for rank, index in enumerate(collaborative_recommendations.index)}

    #  Combinamos las puntuaciones
    final_scores = {}
    all_indices = set(content_scores) | set(collaborative_scores)
    for index in all_indices:
        final_scores[index] = (
            weight_content * content_scores.get(index, 0) +
            weight_knn * collaborative_scores.get(index, 0)
        )
    # Devolvemos un dataframe con ambos resultados.
    final_indices = sorted(final_scores, key=final_scores.get, reverse=True)[:10]
    return original_metadata_df.loc[final_indices]

In [ ]:
# Y hacemos los mismo, el bucle pero de forma hibrida
print("--- TRAYECTORIAS RECOMENDADAS PARA EL PERFIL(HIBRIDO) ---")
for i in range(len(test_user_list_prepared)):
    aux = test_user_list_prepared[i:i+1].copy()
    recommended_profiles = hybrid_recommendation( aux, metadata_processed, metadata)
    print(f"Recomendaciones Usuario N° {i+1}")
    display(recommended_profiles)


--- TRAYECTORIAS RECOMENDADAS PARA EL PERFIL(HIBRIDO) ---
Recomendaciones Usuario N° 1


,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
8706,25,Private,159662,HS-grad,9,Married-civ-spouse,Sales,Own-child,White,Male,0,0,26,United-States,>50K
24111,21,Private,202871,HS-grad,9,Never-married,Machine-op-inspct,Own-child,White,Male,0,0,20,United-States,<=50K
24199,23,Local-gov,197918,Some-college,10,Never-married,Protective-serv,Own-child,White,Male,0,0,40,United-States,>50K
26833,22,Local-gov,273734,HS-grad,9,Never-married,Farming-fishing,Own-child,White,Male,0,0,20,United-States,<=50K
25206,39,Local-gov,236391,HS-grad,9,Married-civ-spouse,Machine-op-inspct,Husband,White,Male,0,0,38,United-States,>50K
8856,19,Private,169853,HS-grad,9,Never-married,Machine-op-inspct,Own-child,White,Male,0,0,24,United-States,<=50K
13322,22,Self-emp-inc,150683,HS-grad,9,Never-married,Machine-op-inspct,Own-child,White,Male,0,0,24,United-States,<=50K
20651,32,Private,209103,HS-grad,9,Married-civ-spouse,Transport-moving,Husband,White,Male,0,0,20,United-States,>50K
24514,22,Private,127768,Some-college,10,Never-married,Other-service,Own-child,White,Male,0,0,32,United-States,>50K
13116,21,Private,231053,HS-grad,9,Never-married,Machine-op-inspct,Own-child,White,Male,0,0,30,United-States,<=50K


Recomendaciones Usuario N° 2


,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
1976,43,Private,130126,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,15024,0,50,United-States,>50K
1928,40,Private,88909,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,15024,0,50,United-States,>50K
2020,42,Private,98211,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,15024,0,45,United-States,>50K
2425,41,Private,151504,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,7688,0,50,United-States,>50K
2747,44,Private,145441,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,7298,0,48,United-States,>50K
2883,46,Private,98637,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,7298,0,50,United-States,>50K


Recomendaciones Usuario N° 3


,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
1103,30,Private,96851,Some-college,10,Never-married,Prof-specialty,Not-in-family,White,Female,0,1719,25,United-States,<=50K
53,42,Private,107276,Some-college,10,Never-married,Exec-managerial,Not-in-family,White,Female,0,2444,40,United-States,>50K
18,22,Private,119592,Assoc-acdm,12,Never-married,Handlers-cleaners,Not-in-family,Black,Male,0,2824,40,United-States,>50K
1085,20,Private,91939,HS-grad,9,Never-married,Adm-clerical,Not-in-family,Black,Female,0,1721,30,United-States,<=50K
1248,18,Private,166889,Some-college,10,Never-married,Handlers-cleaners,Own-child,Black,Female,0,1602,35,United-States,<=50K
1339,45,Private,196584,Assoc-voc,11,Never-married,Prof-specialty,Not-in-family,White,Female,0,1564,40,United-States,>50K
801,27,Private,169117,HS-grad,9,Married-civ-spouse,Adm-clerical,Wife,Black,Female,0,1887,40,United-States,>50K
1097,23,Private,129767,Some-college,10,Never-married,Adm-clerical,Not-in-family,White,Female,0,1721,40,United-States,<=50K
198,27,Local-gov,92431,Some-college,10,Never-married,Protective-serv,Not-in-family,White,Male,0,2231,40,United-States,>50K
307,25,Private,121102,Assoc-acdm,12,Never-married,Tech-support,Not-in-family,White,Female,0,2001,30,United-States,<=50K


# Conclusiones Finales
* El modelo hibrido nos permite recomendar tanto desde una perspectiva "realista" o de vecinos cercanos reales(knn) como mas "optimista" por similitud matematica(cosine_similarity), al fin y al cabo las recomendaciones mas que predecir un resultado exactamentente o valor puntual como lo hacen los modelos de prediccion(Random forest, boosting, etc), ofrecen alternativas que "podrian" funcionar, en estos casos 1 de las alternivas suele funcionar y es mejor que no tener alternativas.